<a href="https://colab.research.google.com/github/mohammedAlkhuzaie/Suha-Ali-Salman/blob/main/Test_car_configuration_Suha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
Test Car Configuration — Self-Contained Version
==================================================


"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set

import pytest


# ---------------------------------------------------------------------------
# Car Configuration implementation (Builder pattern)
# ---------------------------------------------------------------------------

class InvalidConfigurationError(ValueError):
    """Raised when build() is called on a configuration that is incomplete
    or that selects an option the model does not support."""


@dataclass(frozen=True)
class CarModelSpec:
    name: str
    allowed_engines: Set[str]
    allowed_transmissions: Set[str]
    allowed_interior_features: Set[str]
    allowed_exterior_options: Set[str]
    allowed_safety_features: Set[str]
    required_engine: bool = True
    required_transmission: bool = True
    required_safety_features: Set[str] = field(default_factory=set)


@dataclass(frozen=True)
class Car:
    model: str
    engine: Optional[str]
    transmission: Optional[str]
    interior_features: List[str]
    exterior_options: Dict[str, str]
    safety_features: List[str]

    def summary(self) -> str:
        lines = [f"Car order: {self.model}"]
        lines.append(f"  Engine: {self.engine or '—'}")
        lines.append(f"  Transmission: {self.transmission or '—'}")
        lines.append(f"  Interior: {', '.join(self.interior_features) or 'none'}")
        lines.append(
            f"  Exterior: {', '.join(f'{k}={v}' for k, v in self.exterior_options.items()) or 'none'}"
        )
        lines.append(f"  Safety: {', '.join(self.safety_features) or 'none'}")
        return "\n".join(lines)


class CarBuilder:
    """Step-by-step, fluent, flexible builder. Any step may be skipped;
    build() validates the result against the model's spec before returning
    a Car."""

    def __init__(self, spec: CarModelSpec) -> None:
        self._spec = spec
        self._engine: Optional[str] = None
        self._transmission: Optional[str] = None
        self._interior_features: List[str] = []
        self._exterior_options: Dict[str, str] = {}
        self._safety_features: List[str] = []

    def set_engine(self, engine: str) -> "CarBuilder":
        if engine not in self._spec.allowed_engines:
            raise InvalidConfigurationError(
                f"Engine '{engine}' is not offered on {self._spec.name}. "
                f"Choose from {sorted(self._spec.allowed_engines)}."
            )
        self._engine = engine
        return self

    def set_transmission(self, transmission: str) -> "CarBuilder":
        if transmission not in self._spec.allowed_transmissions:
            raise InvalidConfigurationError(
                f"Transmission '{transmission}' is not offered on {self._spec.name}. "
                f"Choose from {sorted(self._spec.allowed_transmissions)}."
            )
        self._transmission = transmission
        return self

    def add_interior_feature(self, feature: str) -> "CarBuilder":
        if feature not in self._spec.allowed_interior_features:
            raise InvalidConfigurationError(
                f"Interior feature '{feature}' is not offered on {self._spec.name}."
            )
        if feature not in self._interior_features:
            self._interior_features.append(feature)
        return self

    def set_exterior_option(self, key: str, value: str) -> "CarBuilder":
        if key not in self._spec.allowed_exterior_options:
            raise InvalidConfigurationError(
                f"Exterior option '{key}' is not offered on {self._spec.name}."
            )
        self._exterior_options[key] = value
        return self

    def add_safety_feature(self, feature: str) -> "CarBuilder":
        if feature not in self._spec.allowed_safety_features:
            raise InvalidConfigurationError(
                f"Safety feature '{feature}' is not offered on {self._spec.name}."
            )
        if feature not in self._safety_features:
            self._safety_features.append(feature)
        return self

    def build(self) -> Car:
        if self._spec.required_engine and self._engine is None:
            raise InvalidConfigurationError(f"{self._spec.name} requires an engine to be selected.")
        if self._spec.required_transmission and self._transmission is None:
            raise InvalidConfigurationError(f"{self._spec.name} requires a transmission to be selected.")
        missing_required_safety = self._spec.required_safety_features - set(self._safety_features)
        if missing_required_safety:
            raise InvalidConfigurationError(
                f"{self._spec.name} requires safety features {sorted(missing_required_safety)} "
                f"which have not been selected."
            )
        return Car(
            model=self._spec.name,
            engine=self._engine,
            transmission=self._transmission,
            interior_features=list(self._interior_features),
            exterior_options=dict(self._exterior_options),
            safety_features=list(self._safety_features),
        )


SEDAN_SPEC = CarModelSpec(
    name="Sedan LX",
    allowed_engines={"V6"},
    allowed_transmissions={"automatic", "manual"},
    allowed_interior_features={"leather seats", "GPS", "sound system"},
    allowed_exterior_options={"color", "rims"},
    allowed_safety_features={"ABS", "airbags", "rear camera"},
    required_safety_features={"ABS", "airbags"},
)

SUV_SPEC = CarModelSpec(
    name="SUV XT",
    allowed_engines={"V6", "V8"},
    allowed_transmissions={"automatic"},
    allowed_interior_features={"leather seats", "GPS", "sound system"},
    allowed_exterior_options={"color", "rims", "sunroof"},
    allowed_safety_features={"ABS", "airbags", "rear camera"},
    required_safety_features={"ABS", "airbags", "rear camera"},
)


# ---------------------------------------------------------------------------
# Tests
# ---------------------------------------------------------------------------

def test_build_valid_sedan():
    car = (
        CarBuilder(SEDAN_SPEC)
        .set_engine("V6")
        .set_transmission("automatic")
        .add_interior_feature("GPS")
        .add_interior_feature("GPS")  # duplicate, should not double up
        .set_exterior_option("color", "black")
        .add_safety_feature("ABS")
        .add_safety_feature("airbags")
        .build()
    )
    assert car.model == "Sedan LX"
    assert car.engine == "V6"
    assert car.transmission == "automatic"
    assert car.interior_features == ["GPS"]
    assert car.exterior_options == {"color": "black"}
    assert set(car.safety_features) == {"ABS", "airbags"}


def test_build_valid_suv_with_more_options():
    car = (
        CarBuilder(SUV_SPEC)
        .set_engine("V8")
        .set_transmission("automatic")
        .add_interior_feature("leather seats")
        .add_interior_feature("sound system")
        .set_exterior_option("sunroof", "panoramic")
        .add_safety_feature("ABS")
        .add_safety_feature("airbags")
        .add_safety_feature("rear camera")
        .build()
    )
    assert car.model == "SUV XT"
    assert car.engine == "V8"
    assert "sunroof" in car.exterior_options


def test_engine_not_offered_on_model_raises():
    with pytest.raises(InvalidConfigurationError):
        CarBuilder(SEDAN_SPEC).set_engine("V8")


def test_transmission_not_offered_raises():
    with pytest.raises(InvalidConfigurationError):
        CarBuilder(SUV_SPEC).set_transmission("manual")


def test_interior_feature_not_offered_raises():
    with pytest.raises(InvalidConfigurationError):
        CarBuilder(SEDAN_SPEC).add_interior_feature("massage seats")


def test_exterior_option_not_offered_raises():
    with pytest.raises(InvalidConfigurationError):
        CarBuilder(SEDAN_SPEC).set_exterior_option("sunroof", "panoramic")


def test_safety_feature_not_offered_raises():
    with pytest.raises(InvalidConfigurationError):
        CarBuilder(SEDAN_SPEC).add_safety_feature("night vision")


def test_build_missing_required_engine_raises():
    with pytest.raises(InvalidConfigurationError):
        CarBuilder(SEDAN_SPEC).set_transmission("automatic").add_safety_feature(
            "ABS"
        ).add_safety_feature("airbags").build()


def test_build_missing_required_transmission_raises():
    with pytest.raises(InvalidConfigurationError):
        CarBuilder(SEDAN_SPEC).set_engine("V6").add_safety_feature("ABS").add_safety_feature(
            "airbags"
        ).build()


def test_build_missing_required_safety_features_raises():
    with pytest.raises(InvalidConfigurationError):
        CarBuilder(SEDAN_SPEC).set_engine("V6").set_transmission("automatic").build()


def test_car_summary_contains_all_fields():
    car = (
        CarBuilder(SEDAN_SPEC)
        .set_engine("V6")
        .set_transmission("manual")
        .add_safety_feature("ABS")
        .add_safety_feature("airbags")
        .build()
    )
    summary = car.summary()
    assert "Sedan LX" in summary
    assert "V6" in summary
    assert "manual" in summary
    assert "none" in summary  # interior/exterior left empty


if __name__ == "__main__":
    # Two execution contexts are supported here:
    #
    # 1. Real script (`python test_car_configuration_standalone.py`):
    #    `__file__` exists and pytest can discover + run the tests in it
    #    normally via pytest.main([__file__]).
    #
    # 2. Pasted into a Colab/Jupyter cell:
    #    `__file__` does not exist, AND even if we skipped that, pytest
    #    discovers tests from files on disk -- it cannot see functions that
    #    only exist as in-memory objects in the notebook's namespace. So in
    #    this case we just call each `test_*` function directly and report
    #    pass/fail ourselves, with no dependency on pytest's file discovery.
    import sys

    try:
        _this_file = __file__
    except NameError:
        _this_file = None

    if _this_file:
        sys.exit(pytest.main([_this_file, "-v"]))
    else:
        test_functions = {
            name: obj
            for name, obj in list(globals().items())
            if name.startswith("test_") and callable(obj)
        }
        passed, failed = 0, []
        for name, fn in test_functions.items():
            try:
                fn()
                passed += 1
                print(f"PASSED  {name}")
            except Exception as exc:  # noqa: BLE001 - surfacing any failure
                failed.append((name, exc))
                print(f"FAILED  {name}  ->  {exc!r}")
        print(f"\n{passed} passed, {len(failed)} failed out of {len(test_functions)} tests")


PASSED  test_build_valid_sedan
PASSED  test_build_valid_suv_with_more_options
PASSED  test_engine_not_offered_on_model_raises
PASSED  test_transmission_not_offered_raises
PASSED  test_interior_feature_not_offered_raises
PASSED  test_exterior_option_not_offered_raises
PASSED  test_safety_feature_not_offered_raises
PASSED  test_build_missing_required_engine_raises
PASSED  test_build_missing_required_transmission_raises
PASSED  test_build_missing_required_safety_features_raises
PASSED  test_car_summary_contains_all_fields

11 passed, 0 failed out of 11 tests
